# 🚀 Customize and Deploy `google/gemma-4-31B-it` on Amazon SageMaker AI
---
This notebook fine-tunes **Google Gemma 4 31B** with QLoRA on a single L40S GPU using the same `sagemaker_code/sft.py` shared by every recipe in this directory.

🔗 Model card: [google/gemma-4-31B-it on Hugging Face](https://huggingface.co/google/gemma-4-31B-it)

---

**Why this notebook is structurally different from sibling notebooks**

Gemma 4 introduces architectural details that the shared training scaffolding doesn't yet handle out of the box. Rather than fork the shared code, the cells below apply a small, **scoped, idempotent set of patches** to `sagemaker_code/requirements.txt`, `sagemaker_code/sft.py`, and `sagemaker_code/utils/merge_adapter_weights.py` before launching the SageMaker job. Every patch:
- creates a `.bak` backup of the original file
- gates its behavior on `"gemma-4" in model_id` so sibling recipes are unaffected
- is reverted in the final cell of this notebook

The four Gemma 4 specifics this handles:

| Quirk | Mitigation |
|---|---|
| Architecture support requires `transformers >= 5.5.0` (shared file pins 4.57.0) | Bumped pins for `transformers / peft / accelerate / trl / datasets / liger-kernel` |
| `Gemma4ClippableLinear` wraps every `nn.Linear`, so PEFT's LoRA dispatcher rejects all targets | Unwrap on language stack (`use_clipped_linears=False`); exclude vision/audio towers |
| Forward pass requires `mm_token_type_ids` even for text-only training | `Gemma4SFTTrainer` subclass injects zeros via `_prepare_inputs` |
| TRL 1.x renamed `ModelConfig.torch_dtype → dtype` | `getattr(.., "dtype") or getattr(.., "torch_dtype")` compat shim |
| `merge_adapter_weights.py` would need ~62 GB VRAM to fp16-merge Gemma 4 31B | Skipped for Gemma 4; bare LoRA adapter is the deliverable (vLLM/LMI consume it directly) |

The PR cover letter discusses upstreaming these into the shared files; until then, this notebook is self-contained.

---


In [ ]:
%pip install -Uq "datasets==4.3.0" \
    "sagemaker==2.253.1"

In [ ]:
import boto3
import sagemaker

In [ ]:
region = boto3.Session().region_name

sess = sagemaker.Session(boto3.Session(region_name=region))

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

role = sagemaker.get_execution_role()

In [ ]:
print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

## Data Preparation for Supervised Fine-tuning

### [Finance-Instruct-500k](https://huggingface.co/datasets/Josephgflowers/Finance-Instruct-500k)

**Finance-Instruct-500k** is a large-scale dataset with about **518,000 entries** focused on the financial domain. It spans topics such as investments, banking, markets, accounting, and corporate finance, offering a wide variety of instruction–response examples.

**Data Format & Structure**:
- Distributed in **JSON** format, with simple conversion to Parquet.  
- Contains a single `train` split with ~518k records.  
- Each record includes:  
  - `system` – context or metadata for the task  
  - `user` – the financial prompt or query  
  - `assistant` – the corresponding response  

**License**: Released under the **Apache-2.0** license.  

**Applications**:

The dataset can support finance-focused tasks such as:  
- Financial question answering  
- Market and investment analysis  
- Topic and sentiment classification  
- Financial entity extraction and document understanding  

In [ ]:
import os
import json
import pprint
from tqdm import tqdm
from datasets import load_dataset

In [ ]:
dataset_parent_path = os.path.join(os.getcwd(), "tmp_cache_local_dataset")
os.makedirs(dataset_parent_path, exist_ok=True)

**Preparing Your Dataset in `messages` format**

This section walks you through creating a conversation-style dataset—the required `messages` format—for directly training LLMs using SageMaker AI.

**What Is the `messages` Format?**

The `messages` format structures instances as chat-like exchanges, wrapping each conversation turn into a role-labeled JSON array. It’s widely used by frameworks like TRL.

Example entry:

```json
{
  "messages": [
    { "role": "system", "content": "You are a helpful assistant." },
    { "role": "user", "content": "How do I bake sourdough?" },
    { "role": "assistant", "content": "First, you need to create a starter by..." }
  ]
}


In [ ]:
dataset_name = "Josephgflowers/Finance-Instruct-500k"
dataset = load_dataset(dataset_name, split="train[:1000]")


In [ ]:
pprint.pp(dataset[0])

In [ ]:
print(f"total number of fine-tunable samples: {len(dataset)}")

In [ ]:
def convert_to_messages(row):
    # Gemma 4's chat template does not accept role="system". Fold any system
    # context into the first user turn instead.
    system_content = "You are a financial reasoning assistant. Read the user's query, restate the key data, and solve step by step. Show calculations clearly, explain any rounding or adjustments, and present the final answer in a concise and professional manner."
    user_content = f"{system_content}\n\n{row['user']}"
    assistant_content = row["assistant"]

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    }


dataset = dataset.map(convert_to_messages, remove_columns=dataset.column_names)


In [ ]:
dataset_filename = os.path.join(dataset_parent_path, f"{dataset_name.replace('/', '--').replace('.', '-')}.jsonl")
dataset.to_json(dataset_filename, lines=True)

#### Upload file to S3

In [ ]:
from sagemaker.s3 import S3Uploader

In [ ]:
data_s3_uri = f"s3://{sess.default_bucket()}/dataset"

uploaded_s3_uri = S3Uploader.upload(
    local_path=dataset_filename,
    desired_s3_uri=data_s3_uri
)
print(f"Uploaded {dataset_filename} to > {uploaded_s3_uri}")

## Fine-Tune LLMs using SageMaker AI

In [ ]:
import time
from sagemaker.modules.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.modules.configs import InputData
from sagemaker.modules.train import ModelTrainer
from getpass import getpass
import yaml
from jinja2 import Template

In [ ]:
MODEL_ID = "google/gemma-4-31B-it"


In [ ]:
hf_token = getpass()

### Training using `PyTorch` Estimator

**Training Using `PyTorch` Estimator**
Leverages the official PyTorch SageMaker container to run a custom training script using the Accelerate and DeepSpeed libraries. This option is ideal for users who want full control over the training pipeline 

---
**Observability**: SageMaker AI has [SageMaker MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) which enables you to accelerate generative AI by making it easier to track experiments and monitor performance of models and AI applications using a single tool.

You can choose to include MLflow as a part of your training workflow to track your model fine-tuning metrics in realtime by simply specifying a **mlflow** tracking arn.

Optionally you can also report to : **tensorboard**, **wandb**.

In [ ]:
MLFLOW_TRACKING_SERVER_ARN = "arn:aws:sagemaker:us-east-1:XXXXXYYYYYZZ:mlflow-tracking-server/<name>" # or None

if MLFLOW_TRACKING_SERVER_ARN:
    reports_to = "mlflow"
else:
    reports_to = "tensorboard"

In [ ]:
job_name = MODEL_ID.replace('/', '--').replace('.', '-')

In [ ]:
if MLFLOW_TRACKING_SERVER_ARN:
    training_env = {
        # mlflow tracking metrics
        "MLFLOW_EXPERIMENT_NAME": f"{job_name}-exp",
        "MLFLOW_TAGS": json.dumps(
            {
                "source.job": "sm-training-jobs", 
                "source.type": "sft", 
                "source.framework": "pytorch"
            }
        ),
        "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_SERVER_ARN,
        "MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING": "true",
        # non tracking metrics - enabled
        "HF_TOKEN": hf_token,
        "FI_EFA_USE_DEVICE_RDMA": "1",
        "NCCL_DEBUG": "INFO",
        "NCCL_SOCKET_IFNAME": "eth0",
        "FI_PROVIDER": "efa",
        "NCCL_PROTO": "simple",
        "NCCL_NET_GDR_LEVEL": "5"
    }
else:
    training_env = {
        # non tracking metrics
        "HF_TOKEN": hf_token,
        "FI_EFA_USE_DEVICE_RDMA": "1",
        "NCCL_DEBUG": "INFO",
        "NCCL_SOCKET_IFNAME": "eth0",
        "FI_PROVIDER": "efa",
        "NCCL_PROTO": "simple",
        "NCCL_NET_GDR_LEVEL": "5"
    }

#### Training strategy - Choose between: `PeFT`/`Spectrum`/`Full-Finetuning`

Here we create a measured mapping of strategy to instance.

In [ ]:
%%writefile sagemaker_code/requirements.txt
# === Gemma 4 31B notebook overrides ===
# Bumps for Gemma 4 architecture support. Sibling notebooks override this file
# in their cell 29 too, so re-running them after this notebook is harmless.
#
# Why each bump:
#   transformers 5.5.0 — Gemma 4 architecture (Gemma4Config) added in 5.5.0
#   peft 0.19.1        — 0.17 hardcodes `from transformers import HybridCache`,
#                        which 5.5 removed; 0.19+ uses lazy imports
#   accelerate 1.13.0  — paired with peft 0.19+ / trl 1.x
#   trl 1.4.0          — first version compatible with transformers >= 4.56;
#                        renames ModelConfig.torch_dtype -> dtype
#   datasets 4.7.0     — required by trl 1.4.0
#   liger-kernel 0.8.0 — 0.6.x imports HybridCache too; 0.8.0 fixes
#
transformers==5.5.0
peft==0.19.1
accelerate==1.13.0
bitsandbytes==0.46.1
datasets==4.7.0
deepspeed==0.17.5
hf-transfer==0.1.8
hf_xet
liger-kernel==0.8.0
lm-eval[api]==0.4.9
kernels>=0.9.0
mlflow
Pillow
safetensors>=0.6.2
sagemaker==2.251.1
sagemaker-mlflow==0.1.0
sentencepiece==0.2.0
tokenizers>=0.21.4
triton
trl==1.4.0
tensorboard
psutil
py7zr
git+https://github.com/triton-lang/triton.git@main#subdirectory=python/triton_kernels
vllm==0.10.1
poetry
yq
psutil
nvidia-ml-py
pyrsmi


In [ ]:
# QLoRA on a single L40S 48 GB. Gemma 4 31B 4-bit weights are ~17 GB.
args = [
    "--config",
    "hf_recipes/google/gemma-4-31B-it--vanilla-peft-qlora.yaml",
    # "--run-eval" # enable this for small models to run eval + tune
]
training_instance_type = "ml.g6e.2xlarge"
training_instance_count = 1


### Apply Gemma 4 patches

The cells below modify three files in `sagemaker_code/` so the shared training scaffolding handles Gemma 4 31B. Every patch:
- gates its behavior on `"gemma-4" in model_id` so sibling recipes are unaffected
- saves a `.bak` of the original
- is reverted by the final cell of this notebook

Re-running any patch cell is idempotent — it restores from `.bak` first, then re-applies.


In [ ]:
# === Patch sagemaker_code/sft.py for Gemma 4 31B ===
# Idempotent: re-running this cell restores from .bak first, then re-applies.
# The final cell of this notebook reverts both patches.
import shutil, pathlib

SFT_PATH = pathlib.Path("sagemaker_code/sft.py")
SFT_BAK = SFT_PATH.with_suffix(".py.bak")

if not SFT_BAK.exists():
    shutil.copy2(SFT_PATH, SFT_BAK)
    print(f"Backed up {SFT_PATH} -> {SFT_BAK}")
else:
    # Restore-then-rewrite for idempotency
    shutil.copy2(SFT_BAK, SFT_PATH)
    print(f"Restored {SFT_PATH} from {SFT_BAK} before re-patching")

src = SFT_PATH.read_text()

# --- Patch 1: TRL 1.x renamed ModelConfig.torch_dtype -> dtype ---
old1 = """    # Determine torch dtype
    if model_args.torch_dtype in ['auto', None]:
        torch_dtype = model_args.torch_dtype
    else:
        torch_dtype = getattr(torch, model_args.torch_dtype)"""
new1 = """    # Determine torch dtype.
    # TRL 1.x renamed ModelConfig.torch_dtype -> dtype; read whichever is present.
    dtype_attr = getattr(model_args, "dtype", None) or getattr(model_args, "torch_dtype", None)
    if dtype_attr in ['auto', None]:
        torch_dtype = dtype_attr
    else:
        torch_dtype = getattr(torch, dtype_attr)"""
assert old1 in src, "Patch 1 anchor missing — sft.py drifted"
src = src.replace(old1, new1)

# --- Patch 2: Gemma 4 exclude_modules + Gemma4ClippableLinear unwrap ---
old2 = """        peft_config = get_peft_config(model_args)
    else:"""
new2 = """        peft_config = get_peft_config(model_args)
        # Gemma 4 special-cases: every linear in the model is wrapped in
        # `Gemma4ClippableLinear(nn.Module)`, so PEFT 0.19's LoRA dispatcher
        # (which only accepts nn.Linear/Conv1d/Conv2d/etc.) rejects every
        # target. For language-stack layers `use_clipped_linears == False`,
        # so the wrapper is a no-op and we can safely unwrap to the inner
        # bnb.nn.Linear4bit, which the existing bnb-4bit dispatcher handles.
        if peft_config is not None and "gemma-4" in (model_args.model_name_or_path or "").lower():
            peft_config.exclude_modules = ["vision_tower", "audio_tower"]
            logger.info(
                f"Gemma 4 detected — exclude_modules={peft_config.exclude_modules}; "
                "will unwrap Gemma4ClippableLinear on language stack before SFTTrainer."
            )
    else:"""
assert old2 in src, "Patch 2 anchor missing"
src = src.replace(old2, new2, 1)

old3 = """    # Load and configure model
    model_kwargs = create_model_kwargs(model_args, training_args, script_args)
    model = load_model(model_args, training_args, script_args, model_kwargs)
    model = configure_model_for_training(model, script_args)"""
new3 = """    # Load and configure model
    model_kwargs = create_model_kwargs(model_args, training_args, script_args)
    model = load_model(model_args, training_args, script_args, model_kwargs)
    model = configure_model_for_training(model, script_args)

    # Gemma 4 PEFT-prep: unwrap Gemma4ClippableLinear on the language model so
    # PEFT's bnb-4bit dispatcher can find a stock bnb.nn.Linear4bit.
    if peft_config is not None and "gemma-4" in (model_args.model_name_or_path or "").lower():
        try:
            from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
            unwrapped = 0
            for parent in model.modules():
                for child_name, child in list(parent.named_children()):
                    if isinstance(child, Gemma4ClippableLinear) and not getattr(child, "use_clipped_linears", False):
                        setattr(parent, child_name, child.linear)
                        unwrapped += 1
            logger.info(f"Gemma 4 unwrap: replaced {unwrapped} non-clipped Gemma4ClippableLinear modules with their inner Linear.")
        except ImportError:
            logger.warning("Could not import Gemma4ClippableLinear — skipping unwrap; PEFT will likely fail.")"""
assert old3 in src, "Patch 3 anchor missing"
src = src.replace(old3, new3, 1)

# --- Patch 4: Gemma4SFTTrainer subclass for mm_token_type_ids ---
old4 = """    # Initialize trainer - use TokenCountingSFTTrainer if FLOPS computation is enabled
    if script_args.compute_flops and FLOPS_METER_AVAILABLE and flops_cb is not None:
        logger.info("Using TokenCountingSFTTrainer for FLOPS calculation")
        trainer = TokenCountingSFTTrainer(
            model=model,
            args=training_args,
            data_collator=collator_fn,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=tokenizer_or_processor,
            peft_config=peft_config,
            callbacks=[flops_cb],
        )
    else:
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            data_collator=collator_fn,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=tokenizer_or_processor,
            peft_config=peft_config,
        )"""
new4 = """    # Gemma 4 text-only mm_token_type_ids: inject via a trainer subclass.
    # Forward pre-hooks don't fire because PEFT calls model.forward() directly,
    # bypassing __call__. Trainer._prepare_inputs runs on every batch.
    _is_gemma4 = "gemma-4" in (model_args.model_name_or_path or "").lower()
    if _is_gemma4:
        import torch as _torch

        class Gemma4SFTTrainer(SFTTrainer):
            def _prepare_inputs(self, inputs):
                inputs = super()._prepare_inputs(inputs)
                if (isinstance(inputs, dict) and "input_ids" in inputs
                        and "mm_token_type_ids" not in inputs and inputs["input_ids"] is not None):
                    inputs["mm_token_type_ids"] = _torch.zeros_like(inputs["input_ids"])
                return inputs

        class Gemma4TokenCountingSFTTrainer(TokenCountingSFTTrainer if FLOPS_METER_AVAILABLE else SFTTrainer):
            def _prepare_inputs(self, inputs):
                inputs = super()._prepare_inputs(inputs)
                if (isinstance(inputs, dict) and "input_ids" in inputs
                        and "mm_token_type_ids" not in inputs and inputs["input_ids"] is not None):
                    inputs["mm_token_type_ids"] = _torch.zeros_like(inputs["input_ids"])
                return inputs

        _SFTCls = Gemma4SFTTrainer
        _TokCls = Gemma4TokenCountingSFTTrainer
        logger.info("Gemma 4 detected — using trainer subclass to inject mm_token_type_ids per batch.")
    else:
        _SFTCls = SFTTrainer
        _TokCls = TokenCountingSFTTrainer if FLOPS_METER_AVAILABLE else SFTTrainer

    # Initialize trainer - use TokenCountingSFTTrainer if FLOPS computation is enabled
    if script_args.compute_flops and FLOPS_METER_AVAILABLE and flops_cb is not None:
        logger.info("Using TokenCountingSFTTrainer for FLOPS calculation")
        trainer = _TokCls(
            model=model,
            args=training_args,
            data_collator=collator_fn,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=tokenizer_or_processor,
            peft_config=peft_config,
            callbacks=[flops_cb],
        )
    else:
        trainer = _SFTCls(
            model=model,
            args=training_args,
            data_collator=collator_fn,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=tokenizer_or_processor,
            peft_config=peft_config,
        )"""
assert old4 in src, "Patch 4 anchor missing"
src = src.replace(old4, new4, 1)

SFT_PATH.write_text(src)
print(f"sft.py patched: 4 hunks applied (dtype-compat, exclude_modules, unwrap, Gemma4SFTTrainer).")


In [ ]:
# === Patch sagemaker_code/utils/merge_adapter_weights.py for Gemma 4 ===
# Gemma 4 31B fp16 (~62 GB) won't fit in a single g6e GPU and the merged
# bnb-4bit + merge_and_unload path doesn't behave well, so for Gemma 4 we
# skip the merge step. The bare LoRA adapter is the deliverable — vLLM,
# DJL Serving / LMI, and Bedrock all consume it directly.
import shutil, pathlib

MRG_PATH = pathlib.Path("sagemaker_code/utils/merge_adapter_weights.py")
MRG_BAK = MRG_PATH.with_suffix(".py.bak")

if not MRG_BAK.exists():
    shutil.copy2(MRG_PATH, MRG_BAK)
    print(f"Backed up {MRG_PATH} -> {MRG_BAK}")
else:
    shutil.copy2(MRG_BAK, MRG_PATH)
    print(f"Restored {MRG_PATH} from {MRG_BAK} before re-patching")

src = MRG_PATH.read_text()

old = """def save_model(model_path_or_id, save_dir, save_tokenizer=True):
  model = AutoPeftModelForCausalLM.from_pretrained(
      model_path_or_id,
      low_cpu_mem_usage=True,
      torch_dtype=torch.float16,
  )  
  # Merge LoRA and base model and save
  model = model.merge_and_unload()        
  model.save_pretrained(save_dir, safe_serialization=True, max_shard_size=\"3GB\")"""
new = """def save_model(model_path_or_id, save_dir, save_tokenizer=True):
  # Detect Gemma 4 via the adapter's base_model_name_or_path so accommodations
  # only fire for that family.
  is_gemma4 = False
  base_model_id = None
  try:
    import json, os
    cfg_path = os.path.join(model_path_or_id, "adapter_config.json")
    if os.path.isfile(cfg_path):
      with open(cfg_path) as f:
        adapter_cfg = json.load(f)
      base_model_id = adapter_cfg.get("base_model_name_or_path") or ""
      is_gemma4 = "gemma-4" in base_model_id.lower()
  except Exception:
    pass

  if is_gemma4:
    print(
        f"[INFO] Gemma 4 detected (base={base_model_id}). Skipping fp16 merge "
        f"(would require ~62GB VRAM). Adapter artifact at {model_path_or_id} "
        f"is the deliverable; vLLM/LMI consume it directly."
    )
    return

  model = AutoPeftModelForCausalLM.from_pretrained(
      model_path_or_id,
      low_cpu_mem_usage=True,
      torch_dtype=torch.float16,
  )
  # Merge LoRA and base model and save
  model = model.merge_and_unload()        
  model.save_pretrained(save_dir, safe_serialization=True, max_shard_size=\"3GB\")"""
assert old in src, "merge_adapter_weights.py anchor missing — drift?"
src = src.replace(old, new, 1)
MRG_PATH.write_text(src)
print("merge_adapter_weights.py patched: skip-fp16-merge for Gemma 4.")


In [ ]:
# === Write Gemma 4 31B QLoRA recipe YAML ===
# Closely mirrors the Gemma 3 27B sibling, with three Gemma 4 specifics:
#   attn_implementation: sdpa  (FA2 caps head_dim<=256; Gemma 4 uses 512)
#   lora_target_modules: [q_proj,k_proj,v_proj,o_proj]  (attention only — name-matched)
#   max_steps: 50  (smoke-test cap; remove for a real training run)
import pathlib, textwrap

YAML_PATH = pathlib.Path("sagemaker_code/hf_recipes/google/gemma-4-31B-it--vanilla-peft-qlora.yaml")
YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
YAML_PATH.write_text(textwrap.dedent("""\
    # Model arguments
    model_name_or_path: google/gemma-4-31B-it
    tokenizer_name_or_path: google/gemma-4-31B-it
    model_revision: main
    # TRL 1.x renamed torch_dtype -> dtype on ModelConfig. The notebook patches
    # sft.py to read either, but using `dtype` here keeps the YAML forward-compat.
    dtype: bfloat16
    # Gemma 4 global-attention layers use head_dim=512; FlashAttention 2 caps at 256.
    # `sdpa` is mandatory for Gemma 4 — do not change without verifying.
    attn_implementation: sdpa
    use_liger: false
    bf16: true
    tf32: true
    output_dir: /opt/ml/output/google/gemma-4-31B-it/peft-qlora/

    # Dataset arguments
    dataset_id_or_path: /opt/ml/input/data/training/Josephgflowers--Finance-Instruct-500k.jsonl
    max_seq_length: 4096
    packing: true

    # LoRA arguments
    use_peft: true
    load_in_4bit: true
    # Gemma 4's Gemma4ClippableLinear wraps every linear; the notebook patches
    # sft.py to unwrap on the language stack. `exclude_modules` for vision/audio
    # is also injected at runtime by the patched sft.py.
    lora_target_modules: ["q_proj", "k_proj", "v_proj", "o_proj"]
    lora_r: 8
    lora_alpha: 16

    # Training arguments
    # Smoke test: 50 steps validates the full stack cheaply (~7 min training,
    # ~17 min total wall clock with image pull). Remove `max_steps` for full runs.
    num_train_epochs: 1
    max_steps: 50
    per_device_train_batch_size: 2
    gradient_accumulation_steps: 2
    gradient_checkpointing: true
    gradient_checkpointing_kwargs:
      use_reentrant: true
    learning_rate: 1.0e-4
    lr_scheduler_type: cosine
    warmup_ratio: 0.1

    # Logging arguments
    logging_strategy: steps
    logging_steps: 2
    report_to:
    - tensorboard
    run_name: gemma-4-31B-it-peft-qlora-Josephgflowers--Finance-Instruct-500k
    save_strategy: "epoch"
    seed: 42
"""))
print(f"Wrote {YAML_PATH}")


In [ ]:
pytorch_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=sess.boto_session.region_name,
    version="2.7.1",
    instance_type=training_instance_type,
    image_scope="training",
)
print(f"Using image: {pytorch_image_uri}")

In [ ]:
source_code = SourceCode(
    source_dir="./sagemaker_code",
    command=f"bash sm_accelerate_train.sh {' '.join(args)}",
)

compute_configs = Compute(
    instance_type=training_instance_type,
    instance_count=training_instance_count,
    keep_alive_period_in_seconds=1800,
    volume_size_in_gb=300
)

base_job_name = f"{job_name}-finetune"
output_path = f"s3://{sess.default_bucket()}/{base_job_name}"

model_trainer = ModelTrainer(
    training_image=pytorch_image_uri,
    source_code=source_code,
    base_job_name=base_job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path,
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=os.path.join(
            output_path,
            dataset_name.replace('/', '--').replace('.', '-'), 
            job_name,
            "checkpoints"
        ), 
        local_path="/opt/ml/checkpoints"
    ),
    role=role,
    environment=training_env
)

In [ ]:
model_trainer.train(
    input_data_config=[
        InputData(
            channel_name="training",
            data_source=uploaded_s3_uri,  
        )
    ], 
    wait=False
)

### Cleanup — restore upstream files

Run this cell after the training job is queued / completed to leave `sagemaker_code/` exactly as it was before this notebook ran. Safe to re-run; no-op if `.bak` files have already been consumed.


In [ ]:
# === Restore upstream files from .bak ===
# Run this after the training job is launched (or after it completes) to
# leave the workspace exactly as it was before this notebook ran.
import shutil, pathlib

for bak in [
    pathlib.Path("sagemaker_code/sft.py.bak"),
    pathlib.Path("sagemaker_code/utils/merge_adapter_weights.py.bak"),
]:
    if bak.exists():
        target = bak.with_suffix("")  # strips .bak suffix
        shutil.copy2(bak, target)
        bak.unlink()
        print(f"Restored {target} from .bak")
    else:
        print(f"No backup at {bak}; skipping")


## Where to find the trained adapter

After the training job completes, the LoRA adapter lands at:

```
s3://<bucket>/<job-name>/output/model.tar.gz
```

The tarball contains `google/gemma-4-31B-it/peft_adapter/` with the standard PEFT layout:
- `adapter_config.json`
- `adapter_model.safetensors`
- the tokenizer + processor configs

For inference, point vLLM (`--enable-lora --lora-modules`) or DJL Serving / LMI (`option.enable_lora=true`) at the extracted adapter directory.
